# Proyecto 2: Sistema Multi-Agente ReAct

Ejecuta el flujo completo en Colab (GPU T4) con **arquitectura hexagonal**: el notebook solo conecta adaptadores reales con el núcleo de `src/`.

Requiere que el código esté subido a la rama `main` del repo de GitHub indicado abajo (el push está pendiente).

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dependencias

In [ ]:
!pip install -q langgraph transformers accelerate bitsandbytes chromadb ddgs datasets

## 3. Código del proyecto

In [ ]:
import os, sys
REPO_URL = "https://github.com/RenzoFelixAponte/Multi-Agent-ReAct.git"
BRANCH = "main"
if not os.path.exists("/content/Multi-Agent-ReAct"):
    !git clone -q --branch {BRANCH} {REPO_URL} /content/Multi-Agent-ReAct
PROJ = "/content/Multi-Agent-ReAct"
sys.path.insert(0, f"{PROJ}/src")

## 4. Adaptadores reales y núcleo

In [ ]:
from adapters.real import MistralLocalLLM, DuckDuckGoSearch, ChromaKB
from adapters.hf_datasets import HFDatasetAdapter, wikiqa_documents
from application.graph import build_flow
from application.naive_rag import naive_rag

llm = MistralLocalLLM("mistralai/Mistral-7B-Instruct-v0.3")  # 4 bits
web = DuckDuckGoSearch()
kb = ChromaKB(path="/content/chroma_db")

## 5. Indexar WikiQA en ChromaDB

In [ ]:
items = HFDatasetAdapter().load("wiki_qa", "validation", limit=300)
kb.index(wikiqa_documents(items))
print(f"{len(items)} consultas, {len(wikiqa_documents(items))} documentos indexados")

## 6. Grafo de agentes

In [ ]:
from IPython.display import Image, display
flow = build_flow(llm, web, kb, max_rounds=3)
try:
    display(Image(flow.get_graph().draw_mermaid_png()))
except Exception as e:  # draw_mermaid_png necesita acceso a mermaid.ink
    print("Diagrama PNG no disponible:", e)
    print(flow.get_graph().draw_mermaid())

## 7. Una consulta con el flujo completo

In [ ]:
state = flow.invoke({"question": items[0].question})
print("Pregunta:", items[0].question)
print("Rondas:", state["rounds"], "| aprobado:", state["verdict"].approved)
print("Respuesta:", state.get("final", "[sin respuesta aprobada]"))

## 8. Multi-agente vs Naive RAG (muestra de 5)

In [ ]:
import pandas as pd

rows = []
for it in items[:5]:
    st = flow.invoke({"question": it.question})
    rows.append({
        "pregunta": it.question,
        "referencia": it.reference,
        "naive_rag": naive_rag(it.question, llm, kb),
        "multiagente": st.get("final", ""),
        "rondas": st["rounds"],
        "aprobado": st["verdict"].approved,
    })
pd.DataFrame(rows)

## 9. Pendiente

- Evaluación sobre 50 consultas con Faithfulness y Answer Relevance (RAGAS o LLM-as-a-Judge).
- Tabla comparativa completa frente a Naive RAG.
- GSM8K y MBPP como consultas adicionales de razonamiento y código.